# Шаг 7. Исправление усечения русских текстов и пересчёт русских результатов

В шаге 6 выяснилось: токенизатор ruGPT-3 в Python обрезал длинные тексты **слева** (оставлял последние 256 токенов), а расширение — **справа** (первые 256). Здесь:
1. обрезка в Python исправляется на правую (как в браузере и как описано в дипломе), признаки ruGPT-3 пересчитываются для русской части M4 и CoAT (≈5 минут);
2. заново проводятся все русские серии: Э1, Э3, Э4, Э5 — рядом печатаются прежние значения;
3. повторяется сравнение с Qwen2.5-0.5B в одинаковых условиях;
4. выбирается русский классификатор для расширения по правилу, заданному заранее (ниже), собирается русская модель для браузера и проверяется в коде расширения — токены должны совпасть на 100 %.

Время: ≈30–40 минут. **Перед запуском:** T4 GPU, «Выполнить все».
**Прислать:** вывод ячеек 3–10, файлы `results_step7.json` и `weights_v3.json`. Архив `extension_assets_v3.zip` — скачать себе.

## 1. Подготовка

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, torch
WORK = '/content/drive/MyDrive/ai-text-detector'
for d in ['/content/code', '/content/js/lib', '/content/new/models', '/content/pack/models', '/content/pack/model']: os.makedirs(d, exist_ok=True)
os.chdir('/content/code'); sys.path.insert(0, '/content/code')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
assert DEVICE == 'cuda', 'Нужна среда с GPU: Среда выполнения → Сменить среду выполнения → T4 GPU'
!pip -q install onnx onnxruntime 2>&1 | tail -1

## 2. Код проекта

In [ ]:
%%writefile /content/code/stylometry.py
"""Стилометрические признаки (та же логика будет реализована в расширении на JavaScript)."""
import re, math, unicodedata
from collections import Counter

RU_FUNC = set("и в во не на я что он с со как а то все она так его но да ты к ко у же вы за бы по только ее её мне было вот от меня еще ещё нет о об из ему теперь когда даже ну вдруг ли если уже или ни быть был него до вас нибудь опять уж вам ведь там потом себя ничего ей может они тут где есть надо ней для мы тебя их чем была сам чтоб без будто чего раз тоже себе под будет ж тогда кто этот того потому этого какой совсем ним здесь этом один почти мой тем чтобы нее неё сейчас были куда зачем всех никогда можно при наконец два об другой хоть после над больше тот через эти нас про всего них какая много разве три эту моя впрочем хорошо свою этой перед иногда лучше чуть том нельзя такой им более всегда конечно всю между".split())
EN_FUNC = set("the of and to a in is that it for on as with was be by this are or from at an but not have has had which they you he she we his her their its can will would there been were if all so no one more also do than about into such these other only some may then".split())

CYR2LAT = {"А":"A","В":"B","Е":"E","К":"K","М":"M","Н":"H","О":"O","Р":"P","С":"C","Т":"T","Х":"X",
           "а":"a","е":"e","о":"o","р":"p","с":"c","у":"y","х":"x"}
LAT2CYR = {v: k for k, v in CYR2LAT.items()}

def cyr_ratio(text):
    letters = [ch for ch in text if ch.isalpha()]
    if not letters: return 0.0
    return sum('Ѐ' <= ch <= 'ӿ' for ch in letters) / len(letters)

def latin_ratio(text):
    letters = [ch for ch in text if ch.isalpha()]
    if not letters: return 0.0
    return sum('a' <= ch.lower() <= 'z' for ch in letters) / len(letters)

def detect_lang(text):
    """'ru' — преобладает кириллица, 'en' — почти только латиница, иначе 'other' (не поддерживается)."""
    if cyr_ratio(text) > 0.5: return 'ru'
    if latin_ratio(text) > 0.9: return 'en'
    return 'other'

def _convert(word, table):
    return ''.join(table.get(ch, ch) for ch in word)

def _mixed(word):
    return any('\u0400' <= ch <= '\u04FF' for ch in word) and any('a' <= ch.lower() <= 'z' for ch in word)

def _fix_word(word, doc_script):
    """Слово из смеси алфавитов приводим к основному алфавиту текста; если это невозможно
    (в слове есть буквы без двойников) — к алфавиту большинства букв слова (защита от гомоглифов)."""
    if not _mixed(word):
        # слово целиком из «чужого» алфавита, но только из букв-двойников (например, кириллическая «а» в английском тексте)
        foreign = CYR2LAT if doc_script == 'lat' else LAT2CYR
        if word and all(ch in foreign for ch in word):
            return _convert(word, foreign)
        return word
    first = _convert(word, LAT2CYR if doc_script == 'cyr' else CYR2LAT)
    if not _mixed(first):
        return first
    cyr = sum('\u0400' <= ch <= '\u04FF' for ch in word)
    lat = sum('a' <= ch.lower() <= 'z' for ch in word)
    return _convert(word, LAT2CYR if cyr > lat else CYR2LAT)

def normalize(text):
    """NFKC, удаление невидимых символов, нормализация пробелов, исправление гомоглифов."""
    t = unicodedata.normalize('NFKC', text)
    t = re.sub(r'[\u200b\u200c\u200d\u2060\ufeff]', '', t)
    t = re.sub(r'[ \t\u00a0]+', ' ', t)
    t = re.sub(r'\s*\n\s*', '\n', t).strip()
    doc_script = 'cyr' if cyr_ratio(t) > 0.5 else 'lat'
    return re.sub(r'\w+', lambda m: _fix_word(m.group(0), doc_script), t)

SENT_SPLIT = re.compile(r'(?<=[.!?…])\s+')
WORD = re.compile(r"[^\W\d_]+(?:[-'’][^\W\d_]+)*", re.UNICODE)

FEATURE_NAMES = ['mean_sent_len', 'cv_sent_len', 'mean_word_len', 'ttr', 'punct_ratio',
                 'func_ratio', 'rep_bigram_ratio', 'rep_trigram_ratio', 'comma_per_sent']

def stylometric_features(text, lang=None):
    lang = lang or detect_lang(text)
    sents = [s for s in SENT_SPLIT.split(text) if s.strip()]
    words = [w.lower() for w in WORD.findall(text)]
    n = max(len(words), 1)
    slens = [len(WORD.findall(s)) for s in sents] or [0]
    mu = sum(slens) / len(slens)
    sd = math.sqrt(sum((x - mu) ** 2 for x in slens) / len(slens))
    func = RU_FUNC if lang == 'ru' else EN_FUNC
    def rep(k):
        grams = [tuple(words[i:i+k]) for i in range(len(words) - k + 1)]
        if not grams: return 0.0
        c = Counter(grams); return sum(v for v in c.values() if v > 1) / len(grams)
    chars = max(len(text), 1)
    return {
        'mean_sent_len': mu,
        'cv_sent_len': sd / mu if mu else 0.0,
        'mean_word_len': sum(map(len, words)) / n,
        'ttr': len(set(words)) / n,
        'punct_ratio': sum(unicodedata.category(ch).startswith('P') for ch in text) / chars,
        'func_ratio': sum(w in func for w in words) / n,
        'rep_bigram_ratio': rep(2),
        'rep_trigram_ratio': rep(3),
        'comma_per_sent': text.count(',') / max(len(sents), 1),
    }

In [ ]:
%%writefile /content/code/metrics_utils.py
"""Метрики качества детектора (раздел 1.6)."""
import numpy as np
from sklearn.metrics import f1_score, roc_auc_score, roc_curve, precision_score, recall_score

def tpr_at_fpr(y, p, fpr_target=0.01):
    fpr, tpr, _ = roc_curve(y, p)
    ok = fpr <= fpr_target
    return float(tpr[ok].max()) if ok.any() else 0.0

def evaluate(y, p, threshold=0.5, n_boot=200, seed=0):
    y, p = np.asarray(y), np.asarray(p); yhat = (p >= threshold).astype(int)
    res = {'n': int(len(y)), 'F1': f1_score(y, yhat), 'Precision': precision_score(y, yhat, zero_division=0),
           'Recall': recall_score(y, yhat), 'FPR': float(((yhat == 1) & (y == 0)).sum() / max((y == 0).sum(), 1)),
           'AUROC': roc_auc_score(y, p), 'TPR@1%FPR': tpr_at_fpr(y, p)}
    rng = np.random.default_rng(seed); boots = []
    for _ in range(n_boot):                       # доверительный интервал AUROC (бутстреп)
        idx = rng.integers(0, len(y), len(y))
        if len(set(y[idx])) == 2: boots.append(roc_auc_score(y[idx], p[idx]))
    res['AUROC_95CI'] = (float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))) if boots else None
    return res

In [ ]:
%%writefile /content/code/baseline_stylometry.py
"""Базовый стилометрический детектор: признаки + логистическая регрессия, отдельно для каждого языка."""
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GroupShuffleSplit
from stylometry import stylometric_features, normalize, FEATURE_NAMES
from metrics_utils import evaluate

def _features_with_format(text, lang):
    t = normalize(text)
    f = stylometric_features(t, lang)
    f['newline_ratio'] = t.count('\n') / max(len(t), 1)   # признак форматирования — только для эксперимента-абляции
    return f

def featurize(df, names=FEATURE_NAMES):
    return np.array([[f[k] for k in names] for f in
                     (_features_with_format(t, l) for t, l in zip(df.text, df.lang))])

def split(df, seed=42):
    """70/15/15 с разбиением по pair_id: человеческий текст и его сгенерированная пара не разделяются."""
    g1 = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=seed)
    tr, rest = next(g1.split(df, groups=df.pair_id))
    rest_df = df.iloc[rest]
    g2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=seed)
    va, te = next(g2.split(rest_df, groups=rest_df.pair_id))
    return df.iloc[tr], rest_df.iloc[va], rest_df.iloc[te]

def run(df, lang, cap_per_class=20000, seed=42, names=FEATURE_NAMES):
    d = df[df.lang == lang]
    d = pd.concat([g.sample(min(len(g), cap_per_class), random_state=seed) for _, g in d.groupby('label')])
    if d.label.nunique() < 2:
        return None, None, None
    tr, va, te = split(d, seed)
    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight='balanced'))
    model.fit(featurize(tr, names), tr.label)
    res = evaluate(te.label.values, model.predict_proba(featurize(te, names))[:, 1])
    res.update({'lang': lang, 'train': len(tr), 'val': len(va), 'test': len(te)})
    coefs = pd.Series(model[-1].coef_[0], index=names).sort_values(key=abs, ascending=False)
    return res, coefs, model

def export_weights(models, path):
    """Сохраняет параметры нормировки и логистической регрессии в JSON для расширения (extension/model/weights.json)."""
    import json
    out = {'version': 'stylometry-v1', 'demo': False, 'langs': {}}
    for lang, m in models.items():
        sc, lr = m[0], m[-1]
        out['langs'][lang] = {'features': FEATURE_NAMES, 'mean': sc.mean_.tolist(), 'scale': sc.scale_.tolist(),
                              'coef': lr.coef_[0].tolist(), 'intercept': float(lr.intercept_[0])}
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(out, f, indent=1)

def balanced_sample(d, cap_per_class, seed=42):
    return pd.concat([g.sample(min(len(g), cap_per_class), random_state=seed) for _, g in d.groupby('label')])

def run_cross(train_df, test_df, lang, cap_per_class=20000, seed=42, names=FEATURE_NAMES):
    """Обучение на одном корпусе и проверка на другом (перенос на незнакомые генераторы, серия Э3)."""
    tr = balanced_sample(train_df[train_df.lang == lang], cap_per_class, seed)
    te = balanced_sample(test_df[test_df.lang == lang], cap_per_class // 4, seed)
    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight='balanced'))
    model.fit(featurize(tr, names), tr.label)
    res = evaluate(te.label.values, model.predict_proba(featurize(te, names))[:, 1])
    res.update({'lang': lang, 'train': len(tr), 'test': len(te)})
    return res

In [ ]:
%%writefile /content/code/data_loading.py
"""Поиск и загрузка данных M4 и RuATD в единую таблицу:
text | label (1 = ИИ, 0 = человек) | lang | source | generator | pair_id"""
import os, json, glob, hashlib
import pandas as pd
from stylometry import detect_lang

HUMAN_KEYS = ['human_text', 'human', 'text_human', 'original_text']
MACHINE_KEYS = ['machine_text', 'machine', 'text_machine', 'generated_text', 'ai_text']
TEXT_KEYS = ['text', 'Text', 'content']
LABEL_KEYS = ['label', 'Label', 'Class', 'class', 'labels']
MODEL_KEYS = ['model', 'Model', 'generator', 'source_model']

def _first(d, keys):
    for k in keys:
        if k in d and d[k] not in (None, ''):
            return k
    return None

def _read_records(path, limit=None):
    recs = []
    with open(path, encoding='utf-8') as f:
        head = f.read(1); f.seek(0)
        if head == '[':                       # обычный JSON-массив
            data = json.load(f); recs = data[:limit] if limit else data
        else:                                 # JSON Lines
            for line in f:
                line = line.strip()
                if not line: continue
                try: recs.append(json.loads(line))
                except json.JSONDecodeError: continue
                if limit and len(recs) >= limit: break
    return recs

def _pid(*parts):
    return hashlib.md5('|'.join(map(str, parts)).encode()).hexdigest()[:12]

def _label_value(v):
    s = str(v).strip().lower()
    if s in ('1', 'm', 'machine', 'ai', 'generated', 'fake', 'true'): return 1
    if s in ('0', 'h', 'human', 'real', 'false'): return 0
    return None

def inspect_files(root, exts=('.jsonl', '.json', '.csv', '.tsv'), max_files=200):
    """Печатает список файлов данных и ключи первой записи — для диагностики."""
    files = [p for p in glob.glob(os.path.join(root, '**', '*'), recursive=True)
             if p.lower().endswith(exts) and os.path.getsize(p) > 0]
    print(f'{root}: найдено файлов данных — {len(files)}')
    for p in sorted(files)[:max_files]:
        size = os.path.getsize(p) / 1e6
        try:
            if p.lower().endswith(('.csv', '.tsv')):
                cols = list(pd.read_csv(p, sep='\t' if p.endswith('.tsv') else ',', nrows=2).columns)
            else:
                r = _read_records(p, limit=1); cols = list(r[0].keys()) if r else []
        except Exception as e:
            cols = [f'ошибка чтения: {e}']
        print(f'  {os.path.relpath(p, root)}  ({size:.1f} МБ)  поля: {cols[:12]}')
    return files

def load_json_dir(root, dataset_name, limit_per_file=None):
    rows = []
    for p in glob.glob(os.path.join(root, '**', '*.json*'), recursive=True):
        try: recs = _read_records(p, limit=limit_per_file)
        except Exception: continue
        fname = os.path.splitext(os.path.basename(p))[0]
        for i, r in enumerate(recs):
            if not isinstance(r, dict): continue
            gen = r.get(_first(r, MODEL_KEYS)) if _first(r, MODEL_KEYS) else fname
            hk, mk = _first(r, HUMAN_KEYS), _first(r, MACHINE_KEYS)
            if hk or mk:                                   # параллельный формат (как в M4)
                pid = _pid(dataset_name, r[hk]) if hk else _pid(dataset_name, fname, i)  # пара = один человеческий текст
                if hk: rows.append((r[hk], 0, fname, 'human', pid))
                if mk:
                    m = r[mk]
                    for j, t in enumerate(m if isinstance(m, list) else [m]):
                        rows.append((t, 1, fname, str(gen), pid))
                continue
            tk, lk = _first(r, TEXT_KEYS), _first(r, LABEL_KEYS)
            if tk and lk is not None and _label_value(r[lk]) is not None:
                rows.append((r[tk], _label_value(r[lk]), fname, str(gen) if _label_value(r[lk]) else 'human', _pid(dataset_name, fname, i)))
    return _finish(rows, dataset_name)

def load_csv_dir(root, dataset_name):
    rows = []
    for p in glob.glob(os.path.join(root, '**', '*.[ct]sv'), recursive=True):
        try: df = pd.read_csv(p, sep='\t' if p.endswith('.tsv') else ',')
        except Exception: continue
        tk = _first({c: 1 for c in df.columns}, TEXT_KEYS)
        lk = _first({c: 1 for c in df.columns}, LABEL_KEYS)
        if not tk or not lk: continue
        fname = os.path.splitext(os.path.basename(p))[0]
        for i, (t, l) in enumerate(zip(df[tk], df[lk])):
            lab = _label_value(l)
            if lab is None:                  # многоклассовая разметка: 'Human' = 0, имя модели = 1
                lab = 0 if str(l).strip().lower() == 'human' else 1
                gen = 'human' if lab == 0 else str(l)
            else:
                gen = 'human' if lab == 0 else 'unknown'
            rows.append((t, lab, fname, gen, _pid(dataset_name, fname, i)))
    return _finish(rows, dataset_name)

# Язык M4 определяем по имени файла: кириллицей пишутся и болгарский, и казахский,
# латиницей — немецкий и индонезийский, поэтому автоопределение по алфавиту здесь ненадёжно.
M4_LANG_BY_PREFIX = {'russian': 'ru', 'arxiv': 'en', 'peerread': 'en', 'reddit': 'en', 'wikihow': 'en', 'wikipedia': 'en'}
BLOOMZ_HUMAN_KEYS = {'arxiv': 'abstract', 'wikihow': 'text', 'wikipedia': 'text', 'reddit': 'text'}

def load_m4(root):
    """Загрузка английской и русской частей M4 с учётом разных форматов файлов."""
    rows = []
    for p in sorted(glob.glob(os.path.join(root, 'data', '*.jsonl'))):
        fname = os.path.splitext(os.path.basename(p))[0]
        prefix, gen_from_name = fname.split('_')[0], fname.split('_')[-1]
        lang = M4_LANG_BY_PREFIX.get(prefix)
        if lang is None:
            continue
        for i, r in enumerate(_read_records(p)):
            gen = str(r.get('model') or gen_from_name)
            human = r.get('human_text') or (r.get(BLOOMZ_HUMAN_KEYS.get(prefix, '')) if 'bloomz' in fname else None)
            machine = r.get('machine_text')
            if prefix == 'peerread' and 'bloomz' in fname:          # отзывы хранятся списками
                human = (r.get('human_reviews') or [None])[0]
                machine = (r.get('bloom_reviews') or [None])[0]
                gen = 'bloomz'
            pid = _pid('M4', human) if isinstance(human, str) else _pid('M4', fname, i)
            if isinstance(human, str): rows.append((human, 0, fname, 'human', pid, lang))
            for t in (machine if isinstance(machine, list) else [machine]):
                if isinstance(t, str): rows.append((t, 1, fname, gen, pid, lang))
    df = pd.DataFrame(rows, columns=['text', 'label', 'source', 'generator', 'pair_id', 'lang'])
    df = df[df.text.str.strip().str.len() > 0]
    df['label'] = df.label.astype(int); df['dataset'] = 'M4'
    return df.drop_duplicates(['text', 'label']).reset_index(drop=True)

def load_coat(config='binary'):
    """Корпус CoAT (опубликованная версия RuATD) с Hugging Face. Столбцы определяются автоматически."""
    from datasets import load_dataset
    ds = load_dataset('RussianNLP/coat', config)
    print(ds)
    frames = []
    for split in ds:
        d = ds[split].to_pandas()
        feats = ds[split].features
        text_col = max([c for c in d.columns if d[c].dtype == object or pd.api.types.is_string_dtype(d[c])],
                       key=lambda c: d[c].astype(str).str.len().mean())
        label_col = None
        for c in d.columns:
            if c == text_col: continue
            names = getattr(feats[c], 'names', None)
            vals = names or sorted(map(str, d[c].dropna().unique()))[:50]
            if any(str(v).lower() in ('h', 'human', 'm', 'machine') for v in vals):
                label_col = c; break
            if c == 'label' and set(map(str, d[c].dropna().unique())) <= {'0', '1'}:
                label_col = c; break    # карточка CoAT: 1 — машинный текст, 0 — человеческий
        if label_col is None:
            print(f'[{split}] столбец с меткой не найден, столбцы: {list(d.columns)}'); continue
        names = getattr(feats[label_col], 'names', None)
        def to_name(v):
            if names and str(v).isdigit() and int(v) < len(names): return names[int(v)]
            return v
        lab = d[label_col].map(to_name)
        lab = lab.astype(str).str.lower()
        if lab.isin(['0', '1']).all():
            y = (lab == '1').astype(int); gen = ['human' if v == 0 else 'unknown' for v in y]
        elif lab.isin(['h', 'm', 'human', 'machine']).all():
            y = lab.isin(['m', 'machine']).astype(int); gen = ['human' if v == 0 else 'unknown' for v in y]
        else:                                     # многоклассовая разметка: human или имя модели
            y = (lab != 'human').astype(int); gen = lab.tolist()
        frames.append(pd.DataFrame({'text': d[text_col], 'label': y, 'source': f'coat_{split}', 'generator': gen,
                                    'pair_id': [_pid('coat', split, i) for i in range(len(d))], 'lang': 'ru'}))
        print(f'[{split}] текст: {text_col}, метка: {label_col}, примеры меток: {lab.unique()[:8]}')
    if not frames: return pd.DataFrame()
    df = pd.concat(frames, ignore_index=True); df['dataset'] = 'RuATD'
    df = df[df.text.astype(str).str.strip().str.len() > 0]
    return df.drop_duplicates(['text', 'label']).reset_index(drop=True)

def _finish(rows, dataset_name):
    df = pd.DataFrame(rows, columns=['text', 'label', 'source', 'generator', 'pair_id'])
    if df.empty: return df
    df = df[df.text.apply(lambda t: isinstance(t, str) and len(t.strip()) > 0)].copy()
    df['lang'] = df.text.apply(detect_lang)
    df['label'] = df.label.astype(int)
    df['dataset'] = dataset_name
    return df.drop_duplicates(['text', 'label']).reset_index(drop=True)

In [ ]:
%%writefile /content/code/lm_features.py
"""Вероятностные признаки по компактной авторегрессионной языковой модели (раздел 2.3, формулы 2.1–2.2).
Для каждого токена считаются: логарифм вероятности, энтропия предсказанного распределения, ранг
фактического токена и дисперсия логарифма вероятности (для признака условной кривизны, как в Fast-DetectGPT)."""
import numpy as np

LM_FEATURE_NAMES = ['mean_lp', 'std_lp', 'mean_entropy', 'mean_log_rank',
                    'rank_1', 'rank_2_10', 'rank_11_100', 'rank_101_1000', 'rank_gt_1000', 'curvature']

def token_stats(texts, model, tok, max_tokens=256, batch_size=8, device='cuda'):
    """Один проход модели по каждому тексту. Возвращает список словарей с массивами по токенам."""
    import torch
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = 'right'
    out = []
    model.eval()
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        enc = tok(batch, return_tensors='pt', truncation=True, max_length=max_tokens,
                  padding=True, add_special_tokens=False).to(device)
        with torch.no_grad():
            logits = model(input_ids=enc.input_ids, attention_mask=enc.attention_mask).logits.float()
        logp = torch.log_softmax(logits[:, :-1], dim=-1)          # прогноз для токенов 2..T
        target = enc.input_ids[:, 1:]
        lp = logp.gather(-1, target.unsqueeze(-1)).squeeze(-1)    # log P(w_i | w_<i)
        p = logp.exp()
        ent = -(p * logp).sum(-1)                                  # энтропия, формула (2.2)
        var = (p * logp.pow(2)).sum(-1) - ent.pow(2)               # дисперсия log p по распределению модели
        rank = (logp > lp.unsqueeze(-1)).sum(-1) + 1               # ранг фактического токена
        mask = enc.attention_mask[:, 1:].bool()
        for b in range(len(batch)):
            m = mask[b]
            out.append({'ids': enc.input_ids[b][enc.attention_mask[b].bool()].cpu().numpy(),
                        'lp': lp[b][m].cpu().numpy(), 'ent': ent[b][m].cpu().numpy(),
                        'var': var[b][m].cpu().numpy(), 'rank': rank[b][m].cpu().numpy()})
        del logits, logp, p
    return out

def aggregate(s, k=None):
    """Признаки по первым k предсказанным токенам (k=None — все)."""
    lp, ent, var, rank = (s[x][:k] if k else s[x] for x in ('lp', 'ent', 'var', 'rank'))
    n = len(lp)
    if n == 0:
        return {f: 0.0 for f in LM_FEATURE_NAMES}
    curv = (lp.sum() + ent.sum()) / np.sqrt(max(var.sum(), 1e-8))   # (∑log p − ∑E[log p]) / √∑Var, E[log p] = −H
    return {
        'mean_lp': float(lp.mean()), 'std_lp': float(lp.std()), 'mean_entropy': float(ent.mean()),
        'mean_log_rank': float(np.log(rank).mean()),
        'rank_1': float((rank == 1).mean()), 'rank_2_10': float(((rank >= 2) & (rank <= 10)).mean()),
        'rank_11_100': float(((rank >= 11) & (rank <= 100)).mean()),
        'rank_101_1000': float(((rank >= 101) & (rank <= 1000)).mean()), 'rank_gt_1000': float((rank > 1000).mean()),
        'curvature': float(curv),
    }


# Дополнительные признаки (шаг 6): вычисляются из тех же величин по токенам, без новых проходов модели.
LM_EXTRA_NAMES = ['lp_q10', 'lp_q50', 'lp_q90', 'std_entropy', 'mean_lp_plus_h',
                  'xent_ratio', 'frac_above_expected', 'mean_var']

def aggregate_ext(s, k=None):
    """lp_q10/50/90 — квантили логарифма вероятности; std_entropy — разброс энтропии;
    mean_lp_plus_h — среднее (log p + H), то есть превышение правдоподобия над ожидаемым в каждой позиции;
    xent_ratio — отношение среднего «удивления» −log p к средней энтропии (одномодельный аналог Binoculars);
    frac_above_expected — доля токенов, для которых log p > −H; mean_var — средняя дисперсия log p."""
    lp, ent, var = (np.asarray(s[x][:k] if k else s[x], dtype=np.float64) for x in ('lp', 'ent', 'var'))
    if len(lp) == 0:
        return {f: 0.0 for f in LM_EXTRA_NAMES}
    return {
        'lp_q10': float(np.quantile(lp, 0.1)), 'lp_q50': float(np.quantile(lp, 0.5)), 'lp_q90': float(np.quantile(lp, 0.9)),
        'std_entropy': float(ent.std()), 'mean_lp_plus_h': float((lp + ent).mean()),
        'xent_ratio': float((-lp).mean() / max(ent.mean(), 1e-6)),
        'frac_above_expected': float((lp > -ent).mean()), 'mean_var': float(var.mean()),
    }

In [ ]:
%%writefile /content/code/hybrid.py
"""Обучение и оценка статистического, стилометрического и гибридного детекторов (разделы 2.3, 3.3)."""
import json
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from stylometry import stylometric_features, normalize, FEATURE_NAMES as STYLO
from lm_features import aggregate, LM_FEATURE_NAMES as LM
from metrics_utils import evaluate

VARIANTS = {'stylometric': STYLO, 'statistical': LM, 'hybrid': STYLO + LM}

def feature_table(texts, langs, stats, k=None, decode=None):
    """Признаки обеих групп. Если задан k, стилометрия считается по тексту первых k+1 токенов."""
    rows = []
    for t, l, s in zip(texts, langs, stats):
        if k is not None and decode is not None:
            t = decode(s['ids'][:k + 1], l)
        f = stylometric_features(t, l)
        f.update(aggregate(s, k))
        rows.append(f)
    return pd.DataFrame(rows)

def fit(X, y, sample_weight=None):
    m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, class_weight='balanced'))
    if sample_weight is None:
        return m.fit(X, y)
    return m.fit(X, y, logisticregression__sample_weight=sample_weight)

def threshold_at_fpr(model, X_val, y_val, fpr):
    """Порог, при котором на валидационной выборке доля ложных срабатываний равна fpr."""
    scores = model.predict_proba(X_val)[:, 1][np.asarray(y_val) == 0]
    return float(np.quantile(scores, 1 - fpr))

def train_eval_all(F_tr, y_tr, F_te, y_te, variants=VARIANTS):
    out, models = {}, {}
    for name, cols in variants.items():
        m = fit(F_tr[cols].values, y_tr)
        out[name] = evaluate(y_te, m.predict_proba(F_te[cols].values)[:, 1])
        models[name] = m
    return out, models

def export(models, thresholds, path, meta):
    """Веса гибридного детектора для расширения."""
    res = {'version': 'hybrid-v1', 'demo': False, 'meta': meta, 'langs': {}}
    for lang, m in models.items():
        sc, lr = m[0], m[-1]
        res['langs'][lang] = {'features': STYLO + LM, 'mean': sc.mean_.tolist(), 'scale': sc.scale_.tolist(),
                              'coef': lr.coef_[0].tolist(), 'intercept': float(lr.intercept_[0]),
                              'thresholds': thresholds[lang]}
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(res, f, indent=1)

In [ ]:
%%writefile /content/code/onnx_utils.py
"""Экспорт языковых моделей в ONNX, квантование INT8 и вычисление признаков через ONNX Runtime (разделы 1.5, 4.1)."""
import os, time, numpy as np

def export_onnx(model, tok, out_dir, opset=17):
    """Экспорт модели (вход: input_ids, attention_mask; выход: logits) и токенизатора в формат для Transformers.js."""
    import torch
    os.makedirs(os.path.join(out_dir, 'onnx'), exist_ok=True)
    model = model.float().cpu().eval()
    model.config.use_cache = False

    class Wrap(torch.nn.Module):
        def __init__(self, m): super().__init__(); self.m = m
        def forward(self, input_ids, attention_mask):
            return self.m(input_ids=input_ids, attention_mask=attention_mask, use_cache=False).logits

    ids = torch.ones(1, 8, dtype=torch.long); mask = torch.ones(1, 8, dtype=torch.long)
    path = os.path.join(out_dir, 'onnx', 'model.onnx')
    kw = dict(input_names=['input_ids', 'attention_mask'], output_names=['logits'], opset_version=opset,
              dynamic_axes={'input_ids': {0: 'batch', 1: 'seq'}, 'attention_mask': {0: 'batch', 1: 'seq'},
                            'logits': {0: 'batch', 1: 'seq'}})
    try:
        torch.onnx.export(Wrap(model), (ids, mask), path, dynamo=False, **kw)
    except TypeError:
        torch.onnx.export(Wrap(model), (ids, mask), path, **kw)
    model.config.save_pretrained(out_dir)
    tok.save_pretrained(out_dir)          # быстрый токенизатор сохраняет tokenizer.json
    return path

def last_matmul_name(path):
    """Имя последнего умножения матриц — выходной слой модели (проекция на словарь)."""
    import onnx
    m = onnx.load(path, load_external_data=False)
    return [n.name for n in m.graph.node if n.op_type in ('MatMul', 'Gemm')][-1]

QUANT_VARIANTS = {
    'q8': {},                                         # по умолчанию: один масштаб на всю матрицу весов
    'q8_per_channel': {'per_channel': True},          # свой масштаб для каждого столбца
    'q8_keep_head': {'keep_head': True},              # выходной слой остаётся в fp32
    'q8_pc_keep_head': {'per_channel': True, 'keep_head': True},
}

def quantize(path_fp32, variant='q8'):
    from onnxruntime.quantization import quantize_dynamic, QuantType
    opts = QUANT_VARIANTS[variant]
    out = path_fp32.replace('model.onnx', f'model_{variant}.onnx')
    kw = {'per_channel': opts.get('per_channel', False)}
    if opts.get('keep_head'):
        kw['nodes_to_exclude'] = [last_matmul_name(path_fp32)]
    quantize_dynamic(path_fp32, out, weight_type=QuantType.QInt8, **kw)
    return out

def size_mb(path):
    return os.path.getsize(path) / 2**20

def ort_token_stats(session, tok, texts, max_tokens=256):
    """То же, что lm_features.token_stats, но через ONNX Runtime на CPU — по одному тексту, как в браузере."""
    out, times = [], []
    for t in texts:
        ids = tok(t, add_special_tokens=False, truncation=True, max_length=max_tokens, return_tensors='np')['input_ids'].astype(np.int64)
        if ids.shape[1] < 2:
            out.append({'ids': ids[0], 'lp': np.array([]), 'ent': np.array([]), 'var': np.array([]), 'rank': np.array([])}); continue
        t0 = time.perf_counter()
        logits = session.run(['logits'], {'input_ids': ids, 'attention_mask': np.ones_like(ids)})[0][0, :-1].astype(np.float64)
        times.append(time.perf_counter() - t0)
        mx = logits.max(1, keepdims=True)
        logp = logits - (np.log(np.exp(logits - mx).sum(1, keepdims=True)) + mx)
        tgt = ids[0, 1:]
        lp = logp[np.arange(len(tgt)), tgt]
        p = np.exp(logp)
        ent = -(p * logp).sum(1)
        var = (p * logp ** 2).sum(1) - ent ** 2
        rank = (logp > lp[:, None]).sum(1) + 1
        out.append({'ids': ids[0], 'lp': lp, 'ent': ent, 'var': var, 'rank': rank})
    return out, times

In [ ]:
%%writefile /content/code/lm_graph.py
"""Вычисление величин по токенам внутри графа модели ONNX (шаг 6).
Раньше модель возвращала логиты — по 50 тысяч чисел на каждый токен, и признаки считались по ним на JavaScript.
Теперь граф сам вычисляет для каждого токена 4 числа: log p, энтропию H, дисперсию log p и ранг."""
import os, time, numpy as np


def make_feature_model(lm):
    import torch

    class FeatureHead(torch.nn.Module):
        def __init__(self, lm):
            super().__init__(); self.lm = lm

        def forward(self, input_ids, attention_mask):
            logits = self.lm(input_ids=input_ids, attention_mask=attention_mask, use_cache=False).logits[:, :-1, :].float()
            target = input_ids[:, 1:].unsqueeze(-1)
            logp = torch.log_softmax(logits, dim=-1)
            lp = torch.gather(logp, -1, target).squeeze(-1)                      # log P(w_i | w_<i)
            p = torch.exp(logp)
            H = -(p * logp).sum(-1)                                              # энтропия
            var = (p * (logp + H.unsqueeze(-1)) ** 2).sum(-1)                    # дисперсия log p (устойчивая форма)
            rank = (logp > lp.unsqueeze(-1)).sum(-1).to(torch.float32) + 1.0     # ранг фактического токена
            return torch.stack([lp, H, var, rank], dim=-1)                      # [batch, T-1, 4]
    return FeatureHead(lm)


def export_feature_graph(lm, tok, out_dir, opset=17):
    """Экспорт графа «модель + величины по токенам» (вход: input_ids, attention_mask; выход: stats) и токенизатора."""
    import torch
    os.makedirs(os.path.join(out_dir, 'onnx'), exist_ok=True)
    lm = lm.float().cpu().eval(); lm.config.use_cache = False
    m = make_feature_model(lm).eval()
    ids = torch.ones(1, 8, dtype=torch.long); mask = torch.ones(1, 8, dtype=torch.long)
    path = os.path.join(out_dir, 'onnx', 'model.onnx')
    kw = dict(input_names=['input_ids', 'attention_mask'], output_names=['stats'], opset_version=opset,
              dynamic_axes={'input_ids': {0: 'batch', 1: 'seq'}, 'attention_mask': {0: 'batch', 1: 'seq'},
                            'stats': {0: 'batch', 1: 'seq_minus_1'}})
    with torch.no_grad():
        try:
            torch.onnx.export(m, (ids, mask), path, dynamo=False, **kw)
        except TypeError:
            torch.onnx.export(m, (ids, mask), path, **kw)
    lm.config.save_pretrained(out_dir); tok.save_pretrained(out_dir)
    return path


def graph_token_stats(session, tok, texts, max_tokens=256):
    """Величины по токенам из графа (по одному тексту, как в браузере). Время — полное: модель + разбор ответа."""
    out, times = [], []
    for t in texts:
        ids = tok(t, add_special_tokens=False, truncation=True, max_length=max_tokens, return_tensors='np')['input_ids'].astype(np.int64)
        if ids.shape[1] < 2:
            out.append({'ids': ids[0], 'lp': np.array([]), 'ent': np.array([]), 'var': np.array([]), 'rank': np.array([])}); continue
        t0 = time.perf_counter()
        st = session.run(['stats'], {'input_ids': ids, 'attention_mask': np.ones_like(ids)})[0][0].astype(np.float64)
        rec = {'ids': ids[0], 'lp': st[:, 0], 'ent': st[:, 1], 'var': st[:, 2], 'rank': st[:, 3].round().astype(np.int64)}
        times.append(time.perf_counter() - t0)
        out.append(rec)
    return out, times


def logits_token_stats_timed(session, tok, texts, max_tokens=256):
    """Прежний способ (граф отдаёт логиты, величины считаются снаружи) — с полным временем на текст."""
    from onnx_utils import ort_token_stats
    out, times = [], []
    for t in texts:
        t0 = time.perf_counter()
        st, _ = ort_token_stats(session, tok, [t], max_tokens)
        times.append(time.perf_counter() - t0); out.append(st[0])
    return out, times

In [ ]:
%%writefile /content/code/hybrid_v2.py
"""Гибридный детектор, версия 2 (шаг 6): дополнительные вероятностные признаки, подбор регуляризации
на валидационной выборке, обучение на нескольких корпусах. Классификатор — по-прежнему логистическая регрессия,
поэтому результат объясним, а в расширении он вычисляется теми же несколькими арифметическими операциями."""
import json
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score
from stylometry import stylometric_features, FEATURE_NAMES as STYLO
from lm_features import aggregate, aggregate_ext, LM_FEATURE_NAMES as LM, LM_EXTRA_NAMES as EXTRA

SETS = {'stylometric': STYLO, 'hybrid_v1': STYLO + LM, 'hybrid_v2': STYLO + LM + EXTRA}
CS = (0.01, 0.03, 0.1, 0.3, 1.0, 3.0)


def feature_table(texts, langs, stats):
    rows = []
    for t, l, s in zip(texts, langs, stats):
        f = stylometric_features(t, l); f.update(aggregate(s)); f.update(aggregate_ext(s)); rows.append(f)
    return pd.DataFrame(rows)


def fit(X, y, C=1.0, sample_weight=None):
    m = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=5000, class_weight='balanced'))
    if sample_weight is None:
        return m.fit(X, y)
    return m.fit(X, y, logisticregression__sample_weight=sample_weight)


def select_c(Xtr, ytr, val_sets, wtr=None, Cs=CS):
    """Подбор коэффициента регуляризации C по среднему AUROC на валидационных выборках val_sets = [(X, y), ...]."""
    best = None
    for C in Cs:
        m = fit(Xtr, ytr, C, wtr)
        score = float(np.mean([roc_auc_score(y, m.predict_proba(X)[:, 1]) for X, y in val_sets]))
        if best is None or score > best[1]:
            best = (C, score, m)
    return best   # (C, средний AUROC на валидации, модель)


def corpus_weights(groups):
    """Веса примеров: суммарный вес каждого корпуса одинаков."""
    groups = np.asarray(groups); w = np.ones(len(groups))
    for g in np.unique(groups):
        w[groups == g] = len(groups) / (len(np.unique(groups)) * (groups == g).sum())
    return w


def threshold_at_fpr(model, X_val, y_val, fpr):
    scores = model.predict_proba(X_val)[:, 1][np.asarray(y_val) == 0]
    return float(np.quantile(scores, 1 - fpr))


def model_json(model, cols, thresholds, C):
    sc, lr = model[0], model[-1]
    return {'features': list(cols), 'mean': sc.mean_.tolist(), 'scale': sc.scale_.tolist(),
            'coef': lr.coef_[0].tolist(), 'intercept': float(lr.intercept_[0]), 'thresholds': thresholds, 'C': C}


def export(langs, path, meta):
    """langs = {lang: {'main': dict, 'stage1': dict}} — основной классификатор и быстрый стилометрический (первая ступень)."""
    res = {'version': 'hybrid-v2', 'demo': False, 'meta': meta, 'langs': {}}
    for lang, d in langs.items():
        res['langs'][lang] = dict(d['main']); res['langs'][lang]['stage1'] = d['stage1']
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(res, f, indent=1, ensure_ascii=False)

In [ ]:
%%writefile /content/code/attacks.py
"""Простые атаки на детекторы (серия Э5): замена букв гомоглифами и внесение опечаток."""
import random, re
from stylometry import CYR2LAT, LAT2CYR

def homoglyph_attack(text, rate=0.15, seed=0):
    """Заменяет долю rate подходящих букв визуально похожими буквами другого алфавита."""
    rnd = random.Random(seed)
    table = {**CYR2LAT, **LAT2CYR}
    return ''.join(table[ch] if ch in table and rnd.random() < rate else ch for ch in text)

def typo_attack(text, rate=0.1, seed=0):
    """В доле rate слов длиной от 4 букв переставляет две соседние буквы."""
    rnd = random.Random(seed)
    def swap(m):
        w = m.group(0)
        if len(w) < 4 or rnd.random() >= rate: return w
        i = rnd.randrange(1, len(w) - 2)
        return w[:i] + w[i + 1] + w[i] + w[i + 2:]
    return re.sub(r'\w+', swap, text)

In [ ]:
%%writefile /content/js/lib/features.js
// Стилометрические признаки — точная копия логики research/stylometry.py (раздел 2.3 диплома).
import { RU_FUNC, EN_FUNC, CYR2LAT, FEATURE_NAMES } from './wordlists.js';
export { FEATURE_NAMES };

const LAT2CYR = Object.fromEntries(Object.entries(CYR2LAT).map(([k, v]) => [v, k]));
const isCyr = (ch) => ch >= 'Ѐ' && ch <= 'ӿ';
const isLatin = (ch) => { const c = ch.toLowerCase(); return c >= 'a' && c <= 'z'; };

export function cyrRatio(text) {
  const letters = [...text].filter((ch) => /\p{L}/u.test(ch));
  if (!letters.length) return 0;
  return letters.filter(isCyr).length / letters.length;
}

export function latinRatio(text) {
  const letters = [...text].filter((ch) => /\p{L}/u.test(ch));
  if (!letters.length) return 0;
  return letters.filter(isLatin).length / letters.length;
}

// 'ru' — преобладает кириллица, 'en' — почти только латиница, иначе 'other' (не поддерживается)
export function detectLang(text) {
  if (cyrRatio(text) > 0.5) return 'ru';
  if (latinRatio(text) > 0.9) return 'en';
  return 'other';
}

const convert = (word, table) => [...word].map((ch) => table[ch] ?? ch).join('');
const isMixed = (word) => [...word].some(isCyr) && [...word].some(isLatin);

// Слово из смеси алфавитов приводим к основному алфавиту текста; если это невозможно —
// к алфавиту большинства букв слова. Слово целиком из «чужих» букв-двойников тоже исправляем.
function fixWord(word, docScript) {
  if (!isMixed(word)) {
    const foreign = docScript === 'lat' ? CYR2LAT : LAT2CYR;
    if (word && [...word].every((ch) => ch in foreign)) return convert(word, foreign);
    return word;
  }
  const first = convert(word, docScript === 'cyr' ? LAT2CYR : CYR2LAT);
  if (!isMixed(first)) return first;
  const chars = [...word];
  const cyr = chars.filter(isCyr).length;
  const lat = chars.filter(isLatin).length;
  return convert(word, cyr > lat ? LAT2CYR : CYR2LAT);
}

// NFKC, удаление невидимых символов, нормализация пробелов, исправление гомоглифов
export function normalize(text) {
  let t = text.normalize('NFKC');
  t = t.replace(/[\u200b\u200c\u200d\u2060\ufeff]/g, '');
  t = t.replace(/[ \t\u00a0]+/g, ' ');
  t = t.replace(/\s*\n\s*/g, '\n').trim();
  const docScript = cyrRatio(t) > 0.5 ? 'cyr' : 'lat';
  return t.replace(/[\p{L}\p{N}_]+/gu, (w) => fixWord(w, docScript));
}

const SENT_SPLIT = /(?<=[.!?…])\s+/u;
const WORD = /\p{L}+(?:[-'’]\p{L}+)*/gu;

export function stylometricFeatures(text, lang) {
  lang = lang || detectLang(text);
  const sents = text.split(SENT_SPLIT).filter((s) => s.trim());
  const words = (text.match(WORD) || []).map((w) => w.toLowerCase());
  const n = Math.max(words.length, 1);
  let slens = sents.map((s) => (s.match(WORD) || []).length);
  if (!slens.length) slens = [0];
  const mu = slens.reduce((a, b) => a + b, 0) / slens.length;
  const sd = Math.sqrt(slens.reduce((a, x) => a + (x - mu) ** 2, 0) / slens.length);
  const func = lang === 'ru' ? RU_FUNC : EN_FUNC;
  const rep = (k) => {
    const total = words.length - k + 1;
    if (total <= 0) return 0;
    const counts = new Map();
    for (let i = 0; i < total; i++) {
      const g = words.slice(i, i + k).join('\u0001');
      counts.set(g, (counts.get(g) || 0) + 1);
    }
    let s = 0; for (const v of counts.values()) if (v > 1) s += v;
    return s / total;
  };
  const chars = Math.max([...text].length, 1);
  const punct = [...text].filter((ch) => /\p{P}/u.test(ch)).length;
  return {
    mean_sent_len: mu,
    cv_sent_len: mu ? sd / mu : 0,
    mean_word_len: words.reduce((a, w) => a + [...w].length, 0) / n,
    ttr: new Set(words).size / n,
    punct_ratio: punct / chars,
    func_ratio: words.filter((w) => func.has(w)).length / n,
    rep_bigram_ratio: rep(2),
    rep_trigram_ratio: rep(3),
    comma_per_sent: (text.split(',').length - 1) / Math.max(sents.length, 1),
  };
}

export function wordCount(text) { return (text.match(WORD) || []).length; }

In [ ]:
%%writefile /content/js/lib/wordlists.js
// Сгенерировано из research/stylometry.py — не редактировать вручную.
export const RU_FUNC = new Set(["а", "без", "более", "больше", "будет", "будто", "бы", "был", "была", "были", "было", "быть", "в", "вам", "вас", "вдруг", "ведь", "во", "вот", "впрочем", "все", "всегда", "всего", "всех", "всю", "вы", "где", "да", "даже", "два", "для", "до", "другой", "его", "ее", "ей", "ему", "если", "есть", "еще", "ещё", "её", "ж", "же", "за", "зачем", "здесь", "и", "из", "или", "им", "иногда", "их", "к", "как", "какая", "какой", "ко", "когда", "конечно", "кто", "куда", "ли", "лучше", "между", "меня", "мне", "много", "может", "можно", "мой", "моя", "мы", "на", "над", "надо", "наконец", "нас", "не", "него", "нее", "ней", "нельзя", "нет", "неё", "ни", "нибудь", "никогда", "ним", "них", "ничего", "но", "ну", "о", "об", "один", "он", "она", "они", "опять", "от", "перед", "по", "под", "после", "потом", "потому", "почти", "при", "про", "раз", "разве", "с", "сам", "свою", "себе", "себя", "сейчас", "со", "совсем", "так", "такой", "там", "тебя", "тем", "теперь", "то", "тогда", "того", "тоже", "только", "том", "тот", "три", "тут", "ты", "у", "уж", "уже", "хорошо", "хоть", "чего", "чем", "через", "что", "чтоб", "чтобы", "чуть", "эти", "этого", "этой", "этом", "этот", "эту", "я"]);
export const EN_FUNC = new Set(["a", "about", "all", "also", "an", "and", "are", "as", "at", "be", "been", "but", "by", "can", "do", "for", "from", "had", "has", "have", "he", "her", "his", "if", "in", "into", "is", "it", "its", "may", "more", "no", "not", "of", "on", "one", "only", "or", "other", "she", "so", "some", "such", "than", "that", "the", "their", "then", "there", "these", "they", "this", "to", "was", "we", "were", "which", "will", "with", "would", "you"]);
export const CYR2LAT = {"А": "A", "В": "B", "Е": "E", "К": "K", "М": "M", "Н": "H", "О": "O", "Р": "P", "С": "C", "Т": "T", "Х": "X", "а": "a", "е": "e", "о": "o", "р": "p", "с": "c", "у": "y", "х": "x"};
export const FEATURE_NAMES = ["mean_sent_len", "cv_sent_len", "mean_word_len", "ttr", "punct_ratio", "func_ratio", "rep_bigram_ratio", "rep_trigram_ratio", "comma_per_sent"];

In [ ]:
%%writefile /content/js/lib/lm.js
// Вероятностные признаки по компактной языковой модели — копия логики research/lm_features.py.
// Токенизатор — Transformers.js, сама модель (ONNX, INT8) — ONNX Runtime Web.
// Модель версии 2 (шаг 6) сама вычисляет для каждого токена 4 числа (log p, H, дисперсию, ранг) — выход «stats».
// Модель версии 1 возвращает логиты (50 тысяч чисел на токен), и величины считаются здесь, в tokenStats.
export const LM_BASE_NAMES = ['mean_lp', 'std_lp', 'mean_entropy', 'mean_log_rank',
  'rank_1', 'rank_2_10', 'rank_11_100', 'rank_101_1000', 'rank_gt_1000', 'curvature'];
export const LM_EXTRA_NAMES = ['lp_q10', 'lp_q50', 'lp_q90', 'std_entropy', 'mean_lp_plus_h',
  'xent_ratio', 'frac_above_expected', 'mean_var'];
export const LM_FEATURE_NAMES = [...LM_BASE_NAMES, ...LM_EXTRA_NAMES];

// Величины по токенам из логитов (модель версии 1): logits — Float32Array размера T×V
export function tokenStats(ids, logits, V) {
  const T = ids.length;
  const lp = [], ent = [], vr = [], rank = [];
  for (let t = 0; t < T - 1; t++) {
    const off = t * V;
    let max = -Infinity;
    for (let v = 0; v < V; v++) if (logits[off + v] > max) max = logits[off + v];
    let sum = 0;
    for (let v = 0; v < V; v++) sum += Math.exp(logits[off + v] - max);
    const lse = max + Math.log(sum);
    const target = Number(ids[t + 1]);
    const lpT = logits[off + target] - lse;
    let H = 0, E2 = 0, r = 1;
    for (let v = 0; v < V; v++) {
      const l = logits[off + v] - lse;
      const p = Math.exp(l);
      H -= p * l; E2 += p * l * l;
      if (l > lpT) r++;
    }
    lp.push(lpT); ent.push(H); vr.push(E2 - H * H); rank.push(r);
  }
  return { lp, ent, var: vr, rank };
}

// Величины по токенам из выхода «stats» (модель версии 2): data — Float32Array размера (T−1)×4
export function statsFromGraph(data, n) {
  const lp = new Array(n), ent = new Array(n), vr = new Array(n), rank = new Array(n);
  for (let i = 0; i < n; i++) {
    lp[i] = data[4 * i]; ent[i] = data[4 * i + 1]; vr[i] = data[4 * i + 2]; rank[i] = Math.round(data[4 * i + 3]);
  }
  return { lp, ent, var: vr, rank };
}

const mean = (a) => a.reduce((x, y) => x + y, 0) / a.length;
const std = (a) => { const m = mean(a); return Math.sqrt(a.reduce((s, x) => s + (x - m) ** 2, 0) / a.length); };
// Квантиль с линейной интерполяцией — как numpy.quantile по умолчанию
export function quantile(a, q) {
  const s = [...a].sort((x, y) => x - y);
  const pos = q * (s.length - 1), lo = Math.floor(pos), hi = Math.ceil(pos);
  return s[lo] + (s[hi] - s[lo]) * (pos - lo);
}

export function aggregate(s) {
  const n = s.lp.length;
  if (!n) return Object.fromEntries(LM_FEATURE_NAMES.map((f) => [f, 0]));
  const m = mean(s.lp);
  const frac = (f) => s.rank.filter(f).length / n;
  const sumLp = s.lp.reduce((a, b) => a + b, 0);
  const sumH = s.ent.reduce((a, b) => a + b, 0);
  const sumVar = s.var.reduce((a, b) => a + b, 0);
  const meanH = sumH / n;
  let above = 0;
  for (let i = 0; i < n; i++) if (s.lp[i] > -s.ent[i]) above++;
  return {
    mean_lp: m, std_lp: std(s.lp), mean_entropy: meanH,
    mean_log_rank: mean(s.rank.map(Math.log)),
    rank_1: frac((r) => r === 1), rank_2_10: frac((r) => r >= 2 && r <= 10),
    rank_11_100: frac((r) => r >= 11 && r <= 100), rank_101_1000: frac((r) => r >= 101 && r <= 1000),
    rank_gt_1000: frac((r) => r > 1000),
    curvature: (sumLp + sumH) / Math.sqrt(Math.max(sumVar, 1e-8)),
    lp_q10: quantile(s.lp, 0.1), lp_q50: quantile(s.lp, 0.5), lp_q90: quantile(s.lp, 0.9),
    std_entropy: std(s.ent), mean_lp_plus_h: (sumLp + sumH) / n,
    xent_ratio: (-sumLp / n) / Math.max(meanH, 1e-6),
    frac_above_expected: above / n, mean_var: sumVar / n,
  };
}

// Загрузка токенизатора и модели. tokenizerLib — модуль Transformers.js, ort — модуль ONNX Runtime Web.
export async function loadLM({ tokenizerLib, ort, modelsUrl, name, file = 'model_quantized.onnx', modelBytes = null,
                             providers = ['wasm'] }) {
  const t0 = performance.now();
  tokenizerLib.env.allowRemoteModels = false;
  tokenizerLib.env.allowLocalModels = true;
  tokenizerLib.env.localModelPath = modelsUrl;
  const tokenizer = await tokenizerLib.AutoTokenizer.from_pretrained(name);
  const source = modelBytes ?? `${modelsUrl}${name}/onnx/${file}`;   // в Node.js (тесты) модель передаётся байтами
  const session = await ort.InferenceSession.create(source, { executionProviders: providers });
  const graphStats = session.outputNames.includes('stats');
  return { tokenizer, session, ort, graphStats, loadMs: performance.now() - t0 };
}

export function tokenize(lm, text, maxTokens = 256) {
  const enc = lm.tokenizer(text, { add_special_tokens: false, truncation: true, max_length: maxTokens });
  return Array.from(enc.input_ids.data, Number);
}

export async function lmFeatures(lm, text, maxTokens = 256) {
  const ids = tokenize(lm, text, maxTokens);
  if (ids.length < 2) return aggregate({ lp: [], ent: [], var: [], rank: [] });
  const T = ids.length;
  const toTensor = (arr) => new lm.ort.Tensor('int64', BigInt64Array.from(arr.map(BigInt)), [1, T]);
  const feeds = { input_ids: toTensor(ids), attention_mask: toTensor(new Array(T).fill(1)) };
  const out = await lm.session.run(feeds);
  if (lm.graphStats) return aggregate(statsFromGraph(out.stats.data, T - 1));
  const logits = out.logits;
  return aggregate(tokenStats(ids, logits.data, logits.dims[2]));
}

In [ ]:
%%writefile /content/js/js_bench.mjs
// Шаг 6: проверка и замер кода расширения в Node.js (ONNX Runtime Web, бэкенд WebAssembly).
// Аргументы: входной JSON {lang: {model, texts}}, выходной JSON, папка с моделями, имя файла модели.
// ORT_THREADS — число потоков WebAssembly (по умолчанию 1).
import fs from 'fs';
import { normalize, stylometricFeatures } from './lib/features.js';
import { loadLM, lmFeatures, tokenize } from './lib/lm.js';

const [inPath, outPath, modelsDir, modelFile = 'model_quantized.onnx'] = process.argv.slice(2);
const threads = Number(process.env.ORT_THREADS || 1);
const ort = await import('onnxruntime-web');
ort.env.wasm.numThreads = threads;
const { AutoTokenizer, env } = await import('@huggingface/transformers');
const input = JSON.parse(fs.readFileSync(inPath, 'utf8'));
const result = { _threads: threads, _file: modelFile };
for (const [lang, { model, texts }] of Object.entries(input)) {
  const bytes = new Uint8Array(fs.readFileSync(`${modelsDir}/${model}/onnx/${modelFile}`));
  const lm = await loadLM({ tokenizerLib: { AutoTokenizer, env }, ort, modelsUrl: modelsDir + '/', name: model, modelBytes: bytes });
  console.log(`${lang}: модель ${modelFile} загружена за ${Math.round(lm.loadMs)} мс, выход: ${lm.session.outputNames.join(',')}, потоков: ${threads}`);
  await lmFeatures(lm, 'warm up text for the model');            // прогрев
  result[lang] = [];
  for (const t of texts) {
    const n = normalize(t);
    const t0 = performance.now();
    const f = { ...stylometricFeatures(n, lang), ...(await lmFeatures(lm, n, 256)) };
    f._ms = performance.now() - t0;
    f._ids = tokenize(lm, n, 256);
    result[lang].push(f);
    if (result[lang].length % 25 === 0) console.log(`  ${lang}: ${result[lang].length} из ${texts.length}`);
  }
}
fs.writeFileSync(outPath, JSON.stringify(result));
console.log('JS: готово');

## 3. Данные и проверка усечения

In [ ]:
import importlib, pickle, time, json, glob, shutil, numpy as np, pandas as pd
import stylometry, metrics_utils, baseline_stylometry, data_loading, lm_features, hybrid, onnx_utils, lm_graph, hybrid_v2, attacks
for m in (stylometry, metrics_utils, baseline_stylometry, data_loading, lm_features, hybrid, onnx_utils, lm_graph, hybrid_v2, attacks): importlib.reload(m)
from stylometry import normalize, stylometric_features
from baseline_stylometry import split
from data_loading import load_coat
from lm_features import token_stats, aggregate, aggregate_ext
from metrics_utils import evaluate
from hybrid_v2 import SETS, fit, select_c, threshold_at_fpr, model_json
from sklearn.metrics import roc_auc_score
from transformers import AutoTokenizer, AutoModelForCausalLM
SEED = 42
df = pd.read_parquet(f'{WORK}/dataset_step1.parquet'); df = df[df.dataset == 'M4']
def bal(d, cap): return pd.concat([g.sample(min(len(g), cap), random_state=SEED) for _, g in d.groupby('label')])
data = {'ru': df[df.lang == 'ru']}
coat = load_coat('binary'); coat['n_words'] = coat.text.astype(str).str.split().str.len()
data['coat'] = bal(coat[coat.n_words >= 30], 3000)
for k, d in data.items():
    d = d.copy(); d['norm'] = d.text.astype(str).map(normalize); data[k] = d.reset_index(drop=True)
RU = 'ai-forever/rugpt3small_based_on_gpt2'
tok = AutoTokenizer.from_pretrained(RU)
print('Сторона усечения в токенизаторе ruGPT-3 по умолчанию:', tok.truncation_side)
sample = data['ru'].norm.iloc[0]
left = tok(sample, truncation=True, max_length=256, add_special_tokens=False)['input_ids']
tok.truncation_side = 'right'
right = tok(sample, truncation=True, max_length=256, add_special_tokens=False)['input_ids']
print('Первые слова при прежней обрезке:', repr(tok.decode(left[:12])))
print('Первые слова при правой обрезке: ', repr(tok.decode(right[:12])))
print('Начало текста:                   ', repr(sample[:70]))
SPL = {}
for k in ['ru', 'coat']:
    d = data[k] if k == 'ru' else data[k].assign(pair_id=data[k].index.astype(str))
    tr, va, te = split(d, SEED); SPL[k] = {'tr': tr.index.values, 'va': va.index.values, 'te': te.index.values}
assert len(SPL['ru']['te']) == 1350 and len(SPL['coat']['te']) == 900, 'Разбиение не совпало — напиши мне'
Y = {k: data[k].label.values.astype(int) for k in data}
OLD = json.load(open(f'{WORK}/results_step3.json', encoding='utf-8'))
RES = {}
def save(): json.dump(RES, open(f'{WORK}/results_step7.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=2, default=str)
def short(r): return f"AUROC={r['AUROC']:.3f} F1={r['F1']:.3f} FPR={r['FPR']:.3f} TPR@1%={r['TPR@1%FPR']:.3f}"

## 4. Признаки ruGPT-3 с правой обрезкой (≈5 минут)

In [ ]:
cache = f'{WORK}/lm_stats_ru_right.pkl'
S = pickle.load(open(cache, 'rb')) if os.path.exists(cache) else {}
lm = AutoModelForCausalLM.from_pretrained(RU).to(DEVICE)
for k in ['ru', 'coat']:
    if k in S: continue
    t0 = time.time(); S[k] = token_stats(list(data[k].norm), lm, tok, max_tokens=256, batch_size=8, device=DEVICE)
    print(f'{k}: {len(S[k])} текстов за {time.time() - t0:.0f} с'); pickle.dump(S, open(cache, 'wb'))
n_tok = np.array([len(tok(t, add_special_tokens=False)['input_ids']) for t in data['ru'].norm])
RES['share_longer_256'] = float((n_tok > 256).mean())
print(f'Доля русских текстов M4 длиннее 256 токенов (на них обрезка влияла): {RES["share_longer_256"]:.1%}')
F = {k: hybrid_v2.feature_table(data[k].norm, ['ru'] * len(data[k]), S[k]) for k in ['ru', 'coat']}
def XY(k, part, cols): idx = SPL[k][part]; return F[k].iloc[idx][cols].values, Y[k][idx]

## 5. Э1 (русский) заново: прежние значения — в скобках

In [ ]:
RES['E1_ru'] = {}; MODELS = {}
names = {'stylometric': 'стилометрический', 'statistical': 'статистический', 'hybrid': 'гибридный'}
cols_of = {'stylometric': SETS['stylometric'], 'statistical': hybrid.VARIANTS['statistical'], 'hybrid': SETS['hybrid_v1']}
for v, cols in cols_of.items():
    m = fit(*XY('ru', 'tr', cols)); MODELS[v] = m
    r = evaluate(XY('ru', 'te', cols)[1], m.predict_proba(XY('ru', 'te', cols)[0])[:, 1])
    RES['E1_ru'][v] = r; old = OLD['E1']['ru'][v]
    print(f'{names[v]:17s} {short(r)}  ДИ {np.round(r["AUROC_95CI"], 3)}   (было AUROC={old["AUROC"]:.3f}, TPR@1%={old["TPR@1%FPR"]:.3f})')
coef = pd.Series(MODELS['hybrid'][-1].coef_[0], index=cols_of['hybrid']).sort_values(key=abs, ascending=False)
RES['E1_ru']['hybrid_top_features'] = coef.head(8).round(3).to_dict(); save()
print('Важнейшие признаки гибрида:', coef.head(8).round(2).to_dict())

## 6. Э3 (перенос на RuATD) заново

In [ ]:
RES['E3'] = {}; RES['E3_in_domain'] = {}
for v, cols in cols_of.items():
    r = evaluate(Y['coat'], MODELS[v].predict_proba(F['coat'][cols].values)[:, 1]); RES['E3'][v] = r
    m = fit(*XY('coat', 'tr', cols)); ri = evaluate(XY('coat', 'te', cols)[1], m.predict_proba(XY('coat', 'te', cols)[0])[:, 1]); RES['E3_in_domain'][v] = ri
    print(f'{names[v]:17s} перенос: {short(r)} (было {OLD["E3"][v]["AUROC"]:.3f}) | обучение на CoAT: AUROC={ri["AUROC"]:.3f} (было {OLD["E3_in_domain"][v]["AUROC"]:.3f})')
save()

## 7. Э4 (длина текста) заново — обучение заново для каждой длины

In [ ]:
RES['E4_ru'] = {}
OLD4 = {'32': (0.661, 0.590, 0.675), '64': (0.733, 0.565, 0.734), '128': (0.777, 0.580, 0.799), '256': (0.818, 0.629, 0.856)}
for k in [32, 64, 128, 256]:
    Fk = hybrid.feature_table(data['ru'].norm, ['ru'] * len(data['ru']), S['ru'], k=k - 1, decode=lambda ids, l: tok.decode(ids))
    row = {}
    for v in ['stylometric', 'statistical', 'hybrid']:
        cols = hybrid.VARIANTS[v]
        m = fit(Fk.iloc[SPL['ru']['tr']][cols].values, Y['ru'][SPL['ru']['tr']])
        row[v] = float(roc_auc_score(Y['ru'][SPL['ru']['te']], m.predict_proba(Fk.iloc[SPL['ru']['te']][cols].values)[:, 1]))
    RES['E4_ru'][k] = row; save()
    print(k, {v: round(a, 3) for v, a in row.items()}, '| было (стил., стат., гибр.):', OLD4[str(k)])

## 8. Э5 (атаки) заново

In [ ]:
from attacks import homoglyph_attack, typo_attack
RES['E5_ru'] = {}; cols = SETS['hybrid_v1']; te = data['ru'].index[SPL['ru']['te']]
sub = pd.concat([g.sample(min(len(g), 300), random_state=SEED) for _, g in data['ru'].loc[te].groupby('label')])
y = sub.label.values
for attack_name, fn in [('без атаки', None), ('гомоглифы 15%', homoglyph_attack), ('опечатки 10%', typo_attack)]:
    raw = sub.text.astype(str).tolist() if fn is None else [fn(t, seed=i) for i, t in enumerate(sub.text.astype(str))]
    for norm_on in ([True] if fn is None else [False, True]):
        texts = [normalize(t) for t in raw] if norm_on else [t.strip() for t in raw]
        st = token_stats(texts, lm, tok, max_tokens=256, batch_size=8, device=DEVICE)
        Fa = pd.DataFrame([{**stylometric_features(t, 'ru'), **aggregate(s)} for t, s in zip(texts, st)])
        r = evaluate(y, MODELS['hybrid'].predict_proba(Fa[cols].values)[:, 1], n_boot=0)
        key = attack_name + ('' if fn is None else (' + нормализация' if norm_on else ' без нормализации'))
        RES['E5_ru'][key] = r; old = OLD['E5']['ru'].get(key, {})
        print(f'{key:38s} AUROC={r["AUROC"]:.3f} Recall={r["Recall"]:.3f} FPR={r["FPR"]:.3f}  (было AUROC={old.get("AUROC", float("nan")):.3f}, FPR={old.get("FPR", float("nan")):.3f})')
save()

## 9. Qwen2.5-0.5B в одинаковых условиях и выбор русского классификатора для расширения
**Правило выбора (задано до запуска):** кандидаты — гибрид v1 и гибрид v2 (C подбирается по валидации M4), оба на признаках ruGPT-3 и обучены только на M4. Выбирается вариант с бо́льшим AUROC на валидационной выборке M4, **но только если** на валидационной части CoAT (чужой корпус для обоих) он не хуже v1 более чем на 0,02; иначе остаётся v1. Qwen в выборе не участвует (модель в 3–5 раз больше) — только сравнение.

In [ ]:
del lm; torch.cuda.empty_cache()
QW = 'Qwen/Qwen2.5-0.5B'; tq = AutoTokenizer.from_pretrained(QW)
print('Сторона усечения в токенизаторе Qwen:', tq.truncation_side); tq.truncation_side = 'right'
qcache = f'{WORK}/lm_stats_qwen05.pkl'
SQ = pickle.load(open(qcache, 'rb')) if os.path.exists(qcache) else {}
if AutoTokenizer.from_pretrained(QW).truncation_side != 'right': SQ = {}      # кэш шага 6 посчитан с другой обрезкой
if any(k not in SQ for k in ['ru', 'coat']):
    try: lmq = AutoModelForCausalLM.from_pretrained(QW, dtype=torch.float16).to(DEVICE)
    except TypeError: lmq = AutoModelForCausalLM.from_pretrained(QW, torch_dtype=torch.float16).to(DEVICE)
    for k in ['ru', 'coat']:
        if k not in SQ: SQ[k] = token_stats(list(data[k].norm), lmq, tq, max_tokens=256, batch_size=4, device=DEVICE)
    pickle.dump(SQ, open(f'{WORK}/lm_stats_qwen05_right.pkl', 'wb')); del lmq; torch.cuda.empty_cache()
FQ = {k: hybrid_v2.feature_table(data[k].norm, ['ru'] * len(data[k]), SQ[k]) for k in ['ru', 'coat']}
def XYq(k, part, cols): idx = SPL[k][part]; return FQ[k].iloc[idx][cols].values, Y[k][idx]
RES['compare'] = {}; CAND = {}
st_only = [c for c in SETS['hybrid_v2'] if c not in SETS['stylometric']]
for lm_name, getXY, Fc in [('ruGPT-3 small', XY, F['coat']), ('Qwen2.5-0.5B', XYq, FQ['coat'])]:
    for setname in ['hybrid_v1', 'hybrid_v2', 'statistical_v2']:
        cols = st_only if setname == 'statistical_v2' else SETS[setname]
        if setname == 'hybrid_v1': C, m = 1.0, fit(*getXY('ru', 'tr', cols))
        else: C, _, m = select_c(*getXY('ru', 'tr', cols), [getXY('ru', 'va', cols)])
        r = {'C': C, 'val_M4': float(roc_auc_score(getXY('ru', 'va', cols)[1], m.predict_proba(getXY('ru', 'va', cols)[0])[:, 1])),
             'val_CoAT': float(roc_auc_score(getXY('coat', 'va', cols)[1], m.predict_proba(getXY('coat', 'va', cols)[0])[:, 1])),
             'test_M4': evaluate(getXY('ru', 'te', cols)[1], m.predict_proba(getXY('ru', 'te', cols)[0])[:, 1]),
             'CoAT_all_6000': evaluate(Y['coat'], m.predict_proba(Fc[cols].values)[:, 1])}
        RES['compare'][f'{setname} | {lm_name}'] = r; save()
        if lm_name.startswith('ruGPT') and setname != 'statistical_v2': CAND[setname] = (m, cols, C, r)
        print(f'{lm_name:14s} {setname:15s} C={C:<5} вал. M4={r["val_M4"]:.3f} вал. CoAT={r["val_CoAT"]:.3f} | тест M4: {short(r["test_M4"])} | весь CoAT: {r["CoAT_all_6000"]["AUROC"]:.3f}')
a, b = CAND['hybrid_v1'][3], CAND['hybrid_v2'][3]
chosen = 'hybrid_v2' if (b['val_M4'] > a['val_M4'] and b['val_CoAT'] >= a['val_CoAT'] - 0.02) else 'hybrid_v1'
m, cols, C, r = CAND[chosen]
Xv, yv = XY('ru', 'va', cols)
thr = {'fpr10': threshold_at_fpr(m, Xv, yv, 0.10), 'fpr02': threshold_at_fpr(m, Xv, yv, 0.02)}
st = SETS['stylometric']; Cs, _, ms = select_c(*XY('ru', 'tr', st), [XY('ru', 'va', st)])
thr_s = {'fpr10': threshold_at_fpr(ms, XY('ru', 'va', st)[0], yv, 0.10), 'fpr02': threshold_at_fpr(ms, XY('ru', 'va', st)[0], yv, 0.02)}
RU_MAIN, RU_STAGE1, RU_MODEL, RU_COLS = model_json(m, cols, thr, C), model_json(ms, st, thr_s, Cs), m, cols
RES['chosen_ru'] = {'name': chosen, 'thresholds': thr, 'stage1_C': Cs}; save()
print(f'\n>>> Для расширения выбран {chosen}; пороги {thr}')

## 10. Русская модель для браузера: экспорт, проверка в Python и в коде расширения (≈10 минут)

In [ ]:
import onnxruntime as ort, subprocess, threading
from onnx_utils import quantize, size_mb
from lm_graph import export_feature_graph, graph_token_stats
out_dir = '/content/new/models/rugpt3small'
fp32 = export_feature_graph(AutoModelForCausalLM.from_pretrained(RU), tok, out_dir)   # tok сохраняется с правой обрезкой
shutil.copy(quantize(fp32, 'q8'), f'{out_dir}/onnx/model_quantized.onnx')
P = f'{out_dir}/onnx/model_quantized.onnx'; print('Объём:', round(size_mb(P), 1), 'МБ')
so = ort.SessionOptions(); so.intra_op_num_threads = 1; sess = ort.InferenceSession(P, so, providers=['CPUExecutionProvider'])
te = data['ru'].index[SPL['ru']['te']]
sub = pd.concat([g.sample(150, random_state=SEED) for _, g in data['ru'].loc[te].groupby('label')])
st_q, t_q = graph_token_stats(sess, tok, sub.norm.tolist())
Fq = pd.DataFrame([{**stylometric_features(t, 'ru'), **aggregate(s), **aggregate_ext(s)} for t, s in zip(sub.norm, st_q)])
RES['browser_python'] = {'auroc_q8_chosen': float(roc_auc_score(sub.label, RU_MODEL.predict_proba(Fq[RU_COLS].values)[:, 1])),
                         'auroc_q8_hybrid_v1': float(roc_auc_score(sub.label, MODELS['hybrid'].predict_proba(Fq[SETS['hybrid_v1']].values)[:, 1])),
                         'ms_median': float(np.median(t_q) * 1000)}
print('Квантованная модель, 300 текстов:', RES['browser_python'], '(в шаге 3 для гибрида: 0,845)'); save()
%cd /content/js
!npm init -y > /dev/null 2>&1
!npm install --ignore-scripts --no-audit --no-fund @huggingface/transformers@3 onnxruntime-web@1 2>&1 | tail -1
pkg = json.load(open('package.json')); pkg['type'] = 'module'; json.dump(pkg, open('package.json', 'w'))
%cd /content/code
s100 = pd.concat([g.sample(50, random_state=SEED + 1) for _, g in data['ru'].loc[te].groupby('label')])
json.dump({'ru': {'model': 'rugpt3small', 'texts': s100.text.astype(str).tolist()}}, open('/content/js/in_ru.json', 'w'), ensure_ascii=False)
if os.path.exists('/content/js/out_ru.json'): os.remove('/content/js/out_ru.json')
p = subprocess.Popen(['node', 'js_bench.mjs', 'in_ru.json', 'out_ru.json', '/content/new/models'], cwd='/content/js',
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
timer = threading.Timer(1200, p.kill); timer.start(); log = []
for line in p.stdout:
    log.append(line.rstrip())
    if 'из' in line or 'загружена' in line: print(line.rstrip())
p.wait(); timer.cancel()
if not os.path.exists('/content/js/out_ru.json'):
    print('!!! Node.js: ошибка'); print('\n'.join(log[-25:]))
else:
    js = json.load(open('/content/js/out_ru.json'))['ru']
    texts = [normalize(t) for t in s100.text.astype(str)]
    st_py, _ = graph_token_stats(sess, tok, texts)
    Fpy = pd.DataFrame([{**stylometric_features(t, 'ru'), **aggregate(s), **aggregate_ext(s)} for t, s in zip(texts, st_py)])
    Fjs = pd.DataFrame([{c: j[c] for c in RU_COLS} for j in js])
    pj, pp = RU_MODEL.predict_proba(Fjs[RU_COLS].values)[:, 1], RU_MODEL.predict_proba(Fpy[RU_COLS].values)[:, 1]
    RES['js_ru'] = {'tokens_equal_share': float(np.mean([list(j['_ids']) == list(map(int, s['ids'])) for j, s in zip(js, st_py)])),
                    'auroc_js': float(roc_auc_score(s100.label, pj)), 'auroc_py': float(roc_auc_score(s100.label, pp)),
                    'max_prob_diff': float(np.abs(pj - pp).max()), 'mean_prob_diff': float(np.abs(pj - pp).mean()),
                    'ms_median': float(np.median([j['_ms'] for j in js]))}
    print('Код расширения:', {k: round(v, 3) for k, v in RES['js_ru'].items()}); save()

## 11. Файлы для расширения

In [ ]:
w = json.load(open(f'{WORK}/weights_v2.json', encoding='utf-8'))
w['version'] = 'hybrid-v3'; w['langs']['ru'] = dict(RU_MAIN); w['langs']['ru']['stage1'] = RU_STAGE1
w['meta']['chosen']['ru'] = RES['chosen_ru']['name'] + ' (ruGPT-3, правая обрезка)'; w['meta']['truncation'] = 'right'
json.dump(w, open(f'{WORK}/weights_v3.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
shutil.copytree(out_dir, '/content/pack/models/rugpt3small', dirs_exist_ok=True, ignore=shutil.ignore_patterns('model.onnx', 'model_q8*.onnx'))
shutil.copy(f'{WORK}/weights_v3.json', '/content/pack/model/weights.json')
if os.path.exists(f'{WORK}/extension_assets_v3.zip'): os.remove(f'{WORK}/extension_assets_v3.zip')
shutil.make_archive(f'{WORK}/extension_assets_v3', 'zip', '/content/pack')
print('Архив:', round(os.path.getsize(f'{WORK}/extension_assets_v3.zip') / 2**20), 'МБ (только русская модель и weights.json)')